# Nạp checkpoint và xem nhãn vs dự đoán

Nạp `student_ttc.pth`, chạy suy luận **streaming từng frame** trên một trip trong tập test rồi vẽ nhãn thật và dự đoán lên video.

Kiến trúc và `forward` được **copy nguyên văn** từ `explore_combined.ipynb` để hai notebook không lệch nhau.

In [6]:
# 1) Trỏ tới thư mục dataset: Colab thì mount Drive, local thì tự dò
import sys, subprocess
from pathlib import Path

DATASET = "hackathon_ttc"   # nhãn min_ttc động học · "deepaccident_ttc" cũng cùng schema

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Nếu import torch chết với 'WinError 1114 ... c10.dll': đó là lỗi Windows cạn static-TLS
# slot, không phải lỗi notebook. Kernel IPython gọi platform.win32_ver() lúc khởi động,
# WMI nạp thêm ~28 DLL và hết slot cho c10.dll. Chạy một lần rồi chọn kernel mới:
#     python scripts/fix_torch_jupyter_windows.py
# Colab và Linux không dính lỗi này.
for pkg, mod in [("torch", "torch"), ("opencv-python-headless", "cv2"),
                 ("matplotlib", "matplotlib")]:
    try:
        __import__(mod)
    except ImportError:
        pip(pkg)
        __import__(mod)

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    # 👇 SỬA cho khớp chỗ bạn để thư mục datasets trên Drive
    DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/data") / DATASET
else:
    here = Path.cwd()
    DATA_ROOT = next((p / "datasets" / DATASET for p in [here, *here.parents]
                      if (p / "datasets" / DATASET / "annotations").is_dir()),
                     here / "datasets" / DATASET)

ANN = DATA_ROOT / "annotations"
if (ANN / "trips.csv").exists():
    print("DATA_ROOT =", DATA_ROOT)
else:
    # Không assert: trên Drive lúc mới chỉ có .zip thì đây là trạng thái BÌNH
    # THƯỜNG, cell 2 sẽ giải nén rồi trỏ lại DATA_ROOT. Nhưng ở local thì gần
    # như luôn là DATASET trỏ sai tên -> liệt kê thẳng cái đang có, đừng để
    # người đọc phải lần ra qua FileNotFoundError ở cell 2.
    ds_dir = next((p / "datasets" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "datasets").is_dir()), None)
    have = sorted(d.name for d in ds_dir.iterdir()
                  if d.is_dir() and (d / "annotations" / "trips.csv").exists()) if ds_dir else []
    print(f"chưa thấy annotations tại {ANN}")
    print(f"  -> dataset có sẵn trong {ds_dir}: {have or 'không có bộ nào'}"
          f"\n  -> sửa DATASET ở đầu cell này (Colab: cell 2 sẽ tìm .zip để giải nén)")


DATA_ROOT = c:\Users\TuHV12\Downloads\hackathon\datasets\hackathon_ttc


In [ ]:
# 2) Nạp annotation (không cần torch Dataset — chỉ cần bảng trip và nhãn TTC)
import torch                      # để trước pandas (xem ghi chú WinError 1114 ở cell 1)
import cv2, numpy as np, pandas as pd

IMG_SIZE = (224, 224)        # (W, H) — phải trùng lúc train

trips = pd.read_csv(ANN / "trips.csv")
frames = pd.read_csv(ANN / "ttc_per_frame.csv")
TTC = {k: g.sort_values("frame_idx").ttc_s.to_numpy(np.float32)
       for k, g in frames.groupby("trip_id")}

# Feature vô hướng cho MỌI trip, dựng bằng đúng hàm của cell kiến trúc.
# Cell này chạy TRƯỚC cell đó nên hàm chưa tồn tại — dời phần dựng bảng
# xuống dưới bằng một lambda gọi lúc cần, thay vì chép lại phép chuẩn hoá.
_trip_meta = trips.set_index("trip_id").to_dict("index")
def feats_of(trip_id):
    g = frames[frames.trip_id == trip_id].sort_values("frame_idx")
    return scalars_from_frames(g, _trip_meta.get(trip_id, {}))

SPLIT = "test"               # chấm trên split model chưa từng thấy
sel = trips[trips.split == SPLIT]
print(f"{len(trips):,} trip · dùng split {SPLIT!r}: {len(sel)} trip "
      f"({int((sel['class'] == 'positive').sum())} positive)")


In [ ]:
# @arch
# 3) ĐẶC TẢ FEATURE PHI-THỊ-GIÁC — một nguồn sự thật duy nhất cho train và infer
#
#    VÌ SAO CÓ CELL NÀY: bộ chấm điểm (Hackathon_Dataset_Redacted) KHÔNG xoá
#    ego.speed_kmh / longitudinal_accel / lateral_accel, metadata.weather,
#    speed_limit_kmh và danh sách targets (id + class). Đã mở T01d.json.gz kiểm
#    từng trường. Model cũ chỉ ăn ảnh nên vứt hết những thứ đó đi.
#
#    Đo trên nhãn thật (AUC dự đoán frame nguy hiểm ttc<=2s bằng MỘT scalar):
#        giảm tốc   practice 0.781 · deepaccident 0.584
#        tốc độ     practice 0.548 · deepaccident 0.579
#    Tức riêng cái giảm tốc đã mạnh hơn F1 0.619 của checkpoint chỉ-ảnh.
#
#    ⚠️ NHƯNG giảm tốc là PHẢN ỨNG của tài xế, không phải nguyên nhân. Học chỉ
#    dựa vào nó thì hệ thống im lặng đúng lúc tài xế KHÔNG kịp phanh — tức đúng
#    lúc cần cảnh báo nhất. Vì vậy cell 6 bật FEAT_DROP: mỗi clip có xác suất
#    20% bị xoá TRẮNG toàn bộ nhánh scalar, buộc nhánh thị giác phải tự đứng được.
#
#    Cell này để TRƯỚC cell dataset và được đánh dấu `# @arch` để
#    scripts/infer_hackathon_videos.py exec lại đúng đoạn này — train và infer
#    dùng CHUNG một hàm chuẩn hoá, không có bản sao thứ hai để lệch nhau.
import numpy as np

FEAT_NAMES = ["speed", "accel", "jerk", "lat_accel", "speed_ratio", "has_limit",
              "n_targets", "d_targets", "cloud", "rain", "wet", "fog", "sun"]

# Bật/tắt từng feature ở ĐÂY. Bỏ tên khỏi list là model không thấy nó nữa —
# dùng để chạy ablation mà không phải sửa dòng code nào khác.
#
#   n_targets / d_targets: practice và redacted đếm actor trong TẦM CẢM BIẾN
#   của bộ sinh (0-15), deepaccident đếm mọi box trong nhãn lidar (hàng chục).
#   Cùng tên, khác nghĩa -> mặc định TẮT. Bật lại thì phải kiểm bằng bảng band
#   tách domain ở cell 7, không phải bằng loss train.
FEAT_USE = ["speed", "accel", "jerk", "lat_accel", "speed_ratio", "has_limit",
            "cloud", "rain", "wet", "fog", "sun"]
FEAT_IDX = [FEAT_NAMES.index(n) for n in FEAT_USE]
N_FEAT = len(FEAT_USE)


def _num(a, default=0.0):
    """Cột CSV -> float array, ô rỗng/NaN/inf thành `default`."""
    a = np.asarray(a, dtype=object)
    out = np.full(len(a), float(default), dtype=np.float32)
    for i, v in enumerate(a):
        try:
            f = float(v)
        except (TypeError, ValueError):
            continue
        if np.isfinite(f):
            out[i] = f
    return out


def build_scalars(speed_kmh, accel, jerk, lat_accel, n_targets,
                  speed_limit_kmh=None, weather=None):
    """(T,) mỗi thứ + 2 giá trị cấp trip -> (T, N_FEAT) float32 đã chuẩn hoá.

    MỌI phép ở đây phải NHÂN QUẢ và KHÔNG dùng thống kê của cả trip (không
    z-score theo trip): lúc chạy thật frame t chỉ biết những gì đã qua. Chia cho
    hằng số cố định + tanh là cách chuẩn hoá duy nhất thoả điều đó.

    tanh chứ không phải clip: clip làm gradient chết hẳn ngoài dải, còn tanh vẫn
    xếp hạng được hai cú phanh cùng "rất gắt" ở mức khác nhau.
    """
    v = _num(speed_kmh) / 3.6                       # km/h -> m/s
    a = _num(accel)
    j = _num(jerk)
    la = _num(lat_accel)
    nt = _num(n_targets)
    lim = float(speed_limit_kmh) if speed_limit_kmh not in (None, "", 0) else 0.0
    w = weather or {}

    T = len(v)
    ones = np.ones(T, dtype=np.float32)
    dnt = np.concatenate([[0.0], np.diff(nt)]).astype(np.float32)

    cols = {
        "speed": v / 20.0,
        "accel": np.tanh(a / 3.0),
        "jerk": np.tanh(j / 10.0),
        # chỉ ĐỘ LỚN: practice lấy lateral_accel từ IMU của CARLA, deepaccident
        # suy từ hướng vector vận tốc. Hai cách chỉ đồng ý về độ lớn, dấu thì
        # không — đưa dấu vào là đưa thêm một dấu hiệu nhận biết domain.
        "lat_accel": np.tanh(np.abs(la) / 2.0),
        "speed_ratio": (np.clip(_num(speed_kmh) / lim, 0, 2) / 2.0 if lim > 0
                        else np.zeros(T, dtype=np.float32)),
        "has_limit": ones * (1.0 if lim > 0 else 0.0),
        "n_targets": np.log1p(nt) / 3.0,
        "d_targets": np.tanh(dnt),
        "cloud": ones * float(w.get("w_cloud", 0.0)) / 100.0,
        "rain": ones * float(w.get("w_rain", 0.0)) / 100.0,
        "wet": ones * float(w.get("w_wet", 0.0)) / 100.0,
        "fog": ones * float(w.get("w_fog", 0.0)) / 100.0,
        # sun_altitude_angle: đêm -90 -> 0.0 · hoàng hôn 15 -> 0.64 · trưa 75 -> 1.0
        "sun": ones * float(np.clip((float(w.get("w_sun_alt", 75.0)) + 90.0) / 165.0, 0, 1)),
    }
    return np.stack([np.asarray(cols[n], dtype=np.float32)
                     for n in FEAT_USE], axis=1)


def scalars_from_frames(df_trip, trip_row):
    """Các dòng ttc_per_frame.csv của MỘT trip (đã sort theo frame_idx) -> (T,N_FEAT)."""
    g = lambda c: df_trip[c] if c in df_trip else np.zeros(len(df_trip))
    return build_scalars(
        g("ego_speed_kmh"), g("ego_accel_mps2"), g("ego_jerk_mps3"),
        g("ego_lat_accel"), g("n_targets"),
        speed_limit_kmh=trip_row.get("speed_limit_kmh", ""),
        weather={k: trip_row.get(k, 0.0) for k in
                 ("w_cloud", "w_rain", "w_wet", "w_fog", "w_sun_alt")},
    )


print(f"{N_FEAT} feature cho model: {', '.join(FEAT_USE)}")
print(f"tắt: {', '.join(n for n in FEAT_NAMES if n not in FEAT_USE) or '(không)'}")


In [ ]:
# @arch
# 6) Student streaming: backbone 2D pretrain + TSM nhân quả + nhánh scalar + TCN
#
#    PHÂN VAI CHO RÕ — Conv2d nhìn ảnh, Conv1d nhìn thời gian, MLP nhìn số đo xe:
#
#      clip (B,T,3,224,224)
#        └─ backbone pretrain ImageNet  [Conv2d]  -> (B·T, D) một vector mỗi frame
#      feat (B,T,N_FEAT)                          -> MLP -> (B·T, SCALAR_DIM)
#        └─ nối lại, reshape (B, D+SCALAR_DIM, T)   T = trục THỜI GIAN
#           └─ CausalTCN  [Conv1d dọc trục thời gian, không dính pixel]
#        └─ head Conv1d(kernel=1) = Linear áp cho từng frame -> p, 1/TTC
#
#    Conv1d ở đây thay GRU: xuất được sang mọi toolchain NPU, cửa sổ ngữ cảnh
#    hữu hạn nên state không trôi qua hàng giờ lái, và latency tất định.
#
#    HAI THỨ MỚI so với bản chỉ-ảnh
#    ------------------------------
#    (1) NHÁNH SCALAR. Ăn 11 feature ở cell 3 (tốc độ, gia tốc, giật, cua, tỉ số
#        vượt tốc, thời tiết). Nối SAU backbone và TRƯỚC TCN, cố ý: TCN mới là
#        chỗ nhìn được lịch sử, nên "đang phanh 0.4 s rồi" là một mẫu thời gian
#        chứ không phải một con số tại chỗ.
#
#    (2) HEAD DEPTH PHỤ. Chỉ tồn tại lúc train. Dự đoán 1/depth trên lưới 28x28
#        từ feature map của backbone, học trên depth GT của practice
#        (datasets/depth_aux). TTC là bài toán MÉT trên giây; ảnh RGB đơn thuần
#        không ràng buộc gì về mét, và deepaccident — 99% dữ liệu train — lại
#        khác domain thị giác. Head này ép backbone giữ thang đo tuyệt đối.
#        `step()` KHÔNG gọi nó, nên nó không tốn một MAC nào lúc chạy thật.
#
#    BACKBONE đổi được thật: TSM gắn bằng forward hook vào các stage do timm khai
#    báo trong `feature_info`, nên không phụ thuộc layout riêng của từng họ model.
#    ĐÃ ĐO trên máy này (tham số · GMAC/frame · train≡stream):
#      mobileone_s1   4.1M · 0.83 · OK      efficientnet_b0   4.6M · 0.39 · OK
#      mobileone_s2   6.6M · 1.30 · OK      efficientnet_b2   8.3M · 0.66 · OK
#      mobileone_s3   8.9M · 1.90 · OK      resnet34         21.7M · 3.66 · OK
#      mobileone_s4  13.7M · 2.99 · OK      repvgg_a2        27.4M · 5.68 · OK
#
#    Giữ họ MobileOne vì nó reparameterize được: lúc train là nhiều nhánh, lúc
#    infer gộp thành MỘT chồng conv thuần — đúng thứ compiler NPU automotive nuốt
#    trơn. s4 nặng gấp 3.6x s1; đo được s1 chạy 4.9 ms/frame trên GPU nên s4 vào
#    khoảng 18 ms, vẫn thừa so với ngân sách 100 ms ở 10 fps.
#
#    ⚠️ Backbone to hơn KHÔNG chữa overfit mà làm nặng thêm. Phần bù nằm ở
#    AUGMENTATION và head depth phụ ở cell 7. Nếu val vẫn xấu đi trong khi loss
#    train giảm, hạ về mobileone_s2 trước khi thử thứ khác.
#
#    BA THỨ ĐỀU NHÂN QUẢ, nếu không thì điểm đo trên tập test là điểm của một
#    model không tồn tại lúc chạy thật: TSM chỉ lấy kênh từ frame TRƯỚC (TSM gốc
#    dịch hai chiều — dùng cả frame tương lai), TCN pad bên trái, ngữ cảnh hữu
#    hạn. Nhánh scalar cũng vậy: accel/jerk ở cell 3 là sai phân LÙI.
import torch.nn as nn
import torch.nn.functional as F

# timm cũ vẫn import được nhưng thiếu mobileone / feature_info dạng dict — so
# phiên bản chứ đừng bắt ImportError (bài học từ transformers 4.43 im lặng thiếu VJEPA2).
import importlib.metadata as _md
from packaging.version import Version as _V

try:
    _need = _V(_md.version("timm")) < _V("1.0.0")
except _md.PackageNotFoundError:
    _need = True
if _need:
    pip("-U", "timm>=1.0.0")
import timm

BACKBONE   = "mobileone_s2"  # xem bảng trên; đổi một dòng là chạy backbone khác
SHIFT_AT   = (1, 2, 3)      # chỉ số stage trong feature_info của timm
SHIFT_FRAC = 8              # 1/8 số kênh lấy từ frame trước
TCN_DIL    = (1, 2, 4, 8)   # receptive field = 1 + 2*(1+2+4+8) = 31 frame ≈ 3.1s @10fps
TCN_DIM    = 256
TTC_CEIL   = 10.0           # 1/TTC dưới 1/10 coi như không có nguy cơ -> TTC = inf
SCALAR_DIM = 64             # bề rộng nhánh scalar sau MLP
USE_SCALARS = True          # False = quay lại model CHỈ-ẢNH, để chạy ablation
AUX_DEPTH  = True           # False = bỏ head depth phụ
DEPTH_HW   = (28, 28)       # lưới đích, phải khớp scripts/build_depth_aux.py
DEPTH_MIN, DEPTH_MAX = 1.0, 80.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Cell này còn được scripts/infer_hackathon_videos.py exec lại NGOÀI notebook,
# nơi cell dataset chưa từng chạy. Đừng để nó phụ thuộc biến của cell khác.
if "IMG_SIZE" not in globals():
    IMG_SIZE = (224, 224)


def to_ttc(inv, ceil=TTC_CEIL):
    """1/TTC -> giây. Dưới ngưỡng thì trả inf thay vì một con số to vô nghĩa."""
    return torch.where(inv > 1.0 / ceil, 1.0 / inv.clamp(min=1e-6),
                       torch.full_like(inv, float("inf")))


def causal_shift(x, t, frac=SHIFT_FRAC):
    """(B*T,C,H,W): 1/8 kênh đầu lấy từ frame TRƯỚC. Frame 0 nhận 0 — khớp đúng
    trạng thái cache rỗng lúc bắt đầu stream."""
    bt, c, h, w = x.shape
    n = max(1, c // frac)
    x = x.view(bt // t, t, c, h, w)
    out = x.clone()
    out[:, 1:, :n] = x[:, :-1, :n]
    out[:, 0, :n] = 0
    return out.view(bt, c, h, w)


class CausalTCN(nn.Module):
    """Depthwise-separable dilated Conv1d dọc THỜI GIAN, pad trái."""

    def __init__(self, d_in, d=TCN_DIM, dil=TCN_DIL):
        super().__init__()
        self.dil = dil
        self.proj = nn.Conv1d(d_in, d, 1)
        self.dw = nn.ModuleList(nn.Conv1d(d, d, 3, dilation=k, groups=d) for k in dil)
        self.pw = nn.ModuleList(nn.Sequential(nn.Conv1d(d, d, 1), nn.BatchNorm1d(d), nn.ReLU())
                                for _ in dil)

    @property
    def receptive_field(self):
        return 1 + 2 * sum(self.dil)

    def forward(self, x):                      # (B,d_in,T) -> (B,d,T)
        h = self.proj(x)
        for k, dw, pw in zip(self.dil, self.dw, self.pw):
            h = h + pw(dw(F.pad(h, (2 * k, 0))))
        return h


class DepthHead(nn.Module):
    """Feature map cuối của backbone -> 1/depth trên lưới DEPTH_HW.

    Xuất qua sigmoid chứ không phải softplus: đích là 1/d với d thuộc
    [1, 80] m, tức nghịch đảo nằm gọn trong [0.0125, 1] — một khoảng CÓ CHẶN.
    softplus không chặn trên, và một head không chặn học đích có chặn thì lúc
    khởi tạo nó phun ra vài chục, gradient bão hoà trước khi kịp học gì.
    """

    def __init__(self, c_in, hw=DEPTH_HW):
        super().__init__()
        self.hw = hw
        self.net = nn.Sequential(
            nn.Conv2d(c_in, 128, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 1, 1))

    def forward(self, fmap):                   # (N,C,h,w) -> (N,H,W) trong [0,1]
        x = F.interpolate(fmap, size=self.hw, mode="bilinear", align_corners=False)
        return torch.sigmoid(self.net(x)).squeeze(1)


class StudentTTC(nn.Module):
    def __init__(self, backbone=BACKBONE, pretrained=True, shift_at=SHIFT_AT,
                 n_scalar=None, aux_depth=None):
        super().__init__()
        self.n_scalar = (0 if not USE_SCALARS else
                         (N_FEAT if n_scalar is None else n_scalar))
        self.net = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        self._mode, self._t = "off", 1
        self.reset()

        with torch.no_grad():                  # đo D thật: MobileNetV3 có num_features
            v, fm = self._frames(torch.zeros(1, 3, 224, 224), want_map=True)
            d, c_map = v.shape[1], fm.shape[1]   # != chiều sau pooling

        names = [f["module"] for f in self.net.feature_info]
        by_name = dict(self.net.named_modules())
        self.shift_pts = [names[i] for i in shift_at if 0 <= i < len(names)]
        for nm in self.shift_pts:
            by_name[nm].register_forward_hook(self._make_hook(nm))

        if self.n_scalar:
            self.scalar = nn.Sequential(
                nn.Linear(self.n_scalar, 64), nn.ReLU(), nn.Linear(64, SCALAR_DIM))
        self.tcn = CausalTCN(d + (SCALAR_DIM if self.n_scalar else 0))
        self.head_cls = nn.Conv1d(TCN_DIM, 1, 1)
        self.head_ttc = nn.Conv1d(TCN_DIM, 1, 1)
        self.head_depth = (DepthHead(c_map)
                           if (AUX_DEPTH if aux_depth is None else aux_depth) else None)

    @property
    def ctx(self):
        return self.tcn.receptive_field

    def _make_hook(self, key):
        def fn(mod, inp, out):
            if self._mode == "batch":
                return causal_shift(out, self._t)
            if self._mode == "stream":
                return self._shift_stream(key, out)
            return None
        return fn

    def _frames(self, x, want_map=False):
        n = self.net
        fm = n.forward_features(x)
        v = n.forward_head(fm, pre_logits=True)
        return (v, fm) if want_map else v

    def _fuse(self, f, feat, b, t):
        """(B*T,D) + (B,T,N_FEAT) -> (B, D+SCALAR_DIM, T) cho TCN."""
        h = f.view(b, t, -1)
        if self.n_scalar:
            if feat is None:
                feat = h.new_zeros(b, t, self.n_scalar)
            s = self.scalar(feat.to(h.dtype).flatten(0, 1)).view(b, t, -1)
            h = torch.cat([h, s], dim=2)
        return h.transpose(1, 2)

    def forward(self, clip, feat=None):        # -> logit (B,T), inv_ttc (B,T)
        b, t = clip.shape[:2]
        self._mode, self._t = "batch", t
        try:
            f = self._frames(clip.flatten(0, 1))
        finally:
            self._mode = "off"
        h = self.tcn(self._fuse(f, feat, b, t))
        return self.head_cls(h).squeeze(1), F.softplus(self.head_ttc(h)).squeeze(1)

    def depth_forward(self, img):
        """(N,3,H,W) ảnh RỜI RẠC -> (N,28,28) 1/depth. Chỉ dùng lúc train.

        Cố ý để _mode = "off": đây là những frame không liên tiếp, bật TSM ở đây
        là trộn kênh của hai cảnh chẳng liên quan gì nhau, và tệ hơn là làm bẩn
        cache streaming đang giữ trạng thái của một trip khác.
        """
        assert self.head_depth is not None, "model dựng với aux_depth=False"
        self._mode = "off"
        _, fm = self._frames(img, want_map=True)
        return self.head_depth(fm)

    # ------------------------- đường streaming -------------------------
    def reset(self):
        """Gọi khi bắt đầu stream mới hoặc khi nguồn video đứt."""
        self._cache, self._fbuf, self._n = {}, None, 0

    def _shift_stream(self, key, x):
        n = max(1, x.shape[1] // SHIFT_FRAC)
        prev, out = self._cache.get(key), x.clone()
        self._cache[key] = x[:, :n].detach().clone()
        out[:, :n] = torch.zeros_like(x[:, :n]) if prev is None else prev
        return out

    @torch.no_grad()
    def step(self, frame, feat=None):          # (1,3,H,W), (1,N_FEAT) -> logit, inv
        self._mode = "stream"
        try:
            f = self._frames(frame)
        finally:
            self._mode = "off"
        g = self._fuse(f, None if feat is None else feat.view(1, 1, -1), 1, 1)
        if self._fbuf is None:
            self._fbuf = torch.zeros(1, g.shape[1], self.ctx,
                                     device=g.device, dtype=g.dtype)
        self._fbuf = torch.cat([self._fbuf[:, :, 1:], g], dim=2)
        # Chỉ đưa vào TCN các frame THẬT. Đẩy cả vector 0 vào là sai: proj có bias
        # nên proj(0) != 0 — phần đệm phải là zero-pad BÊN TRONG conv.
        # (đo được: nhầm chỗ này lệch 285 lần so với đường train)
        self._n = min(self._n + 1, self.ctx)
        h = self.tcn(self._fbuf[:, :, -self._n:])[:, :, -1:]
        return self.head_cls(h).flatten(), F.softplus(self.head_ttc(h)).flatten()

    @torch.no_grad()
    def predict(self, frame, feat=None):
        """Đầu ra dùng thật: (p va chạm, TTC giây). TTC = inf nghĩa là không nguy cơ."""
        logit, inv = self.step(frame, feat)
        return float(torch.sigmoid(logit)), float(to_ttc(inv))


def count_macs(model, shape=(1, 3, 224, 224)):
    """MAC mỗi frame — con số đối chiếu với ngân sách của hộp trên xe.

    Đo qua step() nên head depth phụ KHÔNG được tính: nó không chạy lúc infer.
    """
    total, hooks = [0], []
    def hook(m, inp, out):
        if isinstance(m, (nn.Conv2d, nn.Conv1d)):
            total[0] += out.numel() * m.in_channels // m.groups * int(np.prod(m.kernel_size))
        elif isinstance(m, nn.Linear):
            total[0] += out.numel() * m.in_features
    for mod in model.modules():
        hooks.append(mod.register_forward_hook(hook))
    model.reset()
    with torch.no_grad():
        model.step(torch.zeros(*shape),
                   torch.zeros(1, model.n_scalar) if model.n_scalar else None)
    for h in hooks:
        h.remove()
    model.reset()
    return total[0]


# --- từ đây trở xuống là TỰ KIỂM, chỉ chạy trong notebook --------------------
# scripts/infer_hackathon_videos.py exec lại cell này CHỈ để lấy định nghĩa lớp,
# và nó đặt __name__ = "nb_arch". Không chặn ở đây thì mỗi lần chạy suy luận sẽ
# tải trọng số ImageNet rồi dựng thừa một model chỉ để vứt đi.
if __name__ != "nb_arch":
    student = StudentTTC().eval()
    print(f"{BACKBONE} · ctx {student.ctx} frame ≈ {student.ctx / 10:.1f}s @10fps · "
          f"TSM @ {student.shift_pts}")
    print(f"nhánh scalar: {student.n_scalar or 'TẮT'} feature -> {SCALAR_DIM} chiều · "
          f"head depth phụ: {'BẬT' if student.head_depth is not None else 'tắt'}")

    # --- đường train và đường stream phải trùng nhau -------------------------
    # Chạy KÈM feature: quên nối scalar vào một trong hai đường thì đúng chỗ này
    # phát hiện, chứ không phải sau 32 epoch.
    _clip = torch.rand(1, 8, 3, *IMG_SIZE[::-1])
    _feat = torch.rand(1, 8, student.n_scalar) if student.n_scalar else None
    with torch.no_grad():
        _lb, _ = student(_clip, _feat)
        student.reset()
        _ls = torch.stack([student.step(_clip[:, i],
                                        None if _feat is None else _feat[:, i])[0]
                           for i in range(8)]).flatten()
    _rel = ((_lb[0] - _ls).abs() / _lb[0].abs().clamp(min=1e-3)).max().item()
    student.reset()
    assert _rel < 1e-3, f"train và stream lệch {_rel:.1e} — kiểm tra tính nhân quả"
    print(f"train ≡ stream (sai khác tương đối {_rel:.1e})")

    _p = sum(p.numel() for p in student.parameters()) / 1e6
    _pi = sum(p.numel() for n, p in student.named_parameters()
              if not n.startswith("head_depth")) / 1e6
    print(f"{_p:.1f}M tham số (lúc infer {_pi:.1f}M, bỏ head depth) · "
          f"{count_macs(student) / 1e9:.2f} GMAC/frame "
          f"· ngân sách 10 fps = {count_macs(student) / 1e9 * 10:.1f} GMAC/s")
    _pb, _tt = student.predict(
        torch.rand(1, 3, *IMG_SIZE[::-1]),
        torch.zeros(1, student.n_scalar) if student.n_scalar else None)
    student.reset()
    _txt = "∞ (không nguy cơ)" if _tt == float("inf") else f"{_tt:.1f}s"
    print(f"đầu ra một frame (chưa train): p = {_pb:.3f} · TTC = {_txt}")


In [ ]:
# 8) Nạp checkpoint và xem NHÃN vs DỰ ĐOÁN trên một video
#
#    Chạy bằng đúng đường sẽ chạy trên xe: step() từng frame, có state, không
#    forward cả clip. Nếu đo bằng forward clip thì con số đẹp hơn thực tế vì
#    frame đầu đã có sẵn ngữ cảnh.
#
#    KHÔNG reparameterize ở đây. reparameterize_model() deepcopy model, mà hook
#    TSM là closure bắt `self` của bản GỐC — bản copy gọi step() sẽ set _mode
#    trên chính nó trong khi hook vẫn đọc _mode của bản gốc, nên TSM im lặng
#    không hoạt động. Reparameterize chỉ để đo tốc độ; ở đây cần đúng số.
#
#    Cell chạy được với CẢ HAI loại nhãn — chúng khác NGHĨA, không chỉ khác cột:
#      combined_ego    ttc = thời gian tới vụ va chạm ĐÃ XẢY RA trong clip.
#                      Mỗi trip có đúng một mốc -> cột collision_frame.
#                      ttc = -1 nghĩa là trip này không có tai nạn.
#      hackathon_ttc   ttc = min_ttc động học, tính lại từng frame.
#                      KHÔNG có "frame va chạm" duy nhất nên KHÔNG có cột đó.
#                      ttc = -1 nghĩa là lúc này không có target nào trong
#                      collision cone, tức AN TOÀN — không phải "thiếu nhãn".
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 200

CKPT_PATH = next((p / "student_ttc.pth" for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "student_ttc.pth").exists()), None)
assert CKPT_PATH, "không thấy student_ttc.pth"

ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print(f"checkpoint: {CKPT_PATH.name} · backbone {ck['backbone']} · "
      f"shift_at {ck['shift_at']} · tcn_dil {ck['tcn_dil']} · "
      f"POS_TTC {ck['pos_ttc']} · NEG_TTC {ck['neg_ttc']} · "
      f"nhãn {ck.get('label', 'collision_countdown')}")

# aux_depth=False + n_scalar LẤY TỪ CHECKPOINT, giống scripts/infer_hackathon_videos.py.
# Head depth chỉ là công cụ regularize lúc train, predict() không bao giờ gọi nó.
# Và số chiều nhánh scalar phải theo file trọng số chứ không theo cấu hình cell 3:
# chỉ file đó mới biết nó đã được train với gì, nên checkpoint chỉ-ảnh cũ
# (không có khoá n_scalar -> 0) vẫn nạp được ở đây.
model = StudentTTC(backbone=ck["backbone"], pretrained=False,
                   shift_at=ck["shift_at"], aux_depth=False,
                   n_scalar=ck.get("n_scalar", 0)).to(DEVICE)
_state = {k: v for k, v in ck["state"].items() if not k.startswith("head_depth.")}
missing, unexpected = model.load_state_dict(_state, strict=False)
assert not missing and not unexpected, (missing[:5], unexpected[:5])
# Số chiều khớp là chưa đủ: đổi THỨ TỰ FEAT_USE mà giữ nguyên số lượng thì model
# nhận đúng 11 số nhưng sai ý nghĩa từng ô — không lỗi, chỉ tệ đi khó hiểu.
_fck = ck.get("feat_use")
assert not (_fck and list(_fck) != list(FEAT_USE)), (
    f"FEAT_USE lúc train {list(_fck)} khác cell 3 hiện tại {list(FEAT_USE)}")
print(f"nhánh scalar: {model.n_scalar or 'tắt'} feature")
model.eval()
print(f"nạp {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M tham số, khớp hết key")

# --- chọn 1 trip trong tập TEST (model chưa từng thấy) ----------------------
SEED = None
ONLY_CLASS = "positive"

pool = sel
if ONLY_CLASS:
    pool = pool[pool["class"] == ONLY_CLASS]
row = pool.iloc[int(np.random.default_rng(SEED).integers(len(pool)))]
gt = TTC[row.trip_id]

IS_MIN_TTC = str(row.get("ttc_label", "")) == "min_ttc"
NEG_TEXT = "∞ an toàn" if IS_MIN_TTC else "—"

# --- chạy streaming, thu dự đoán từng frame --------------------------------
ft = feats_of(row.trip_id) if model.n_scalar else None
cap = cv2.VideoCapture(str(DATA_ROOT / row.video))
model.reset()
imgs, p_hat, ttc_hat = [], [], []
k = 0
while True:
    ok, im = cap.read()
    if not ok or k >= len(gt):
        break
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    imgs.append(rgb)
    x = cv2.resize(rgb, IMG_SIZE)
    x = torch.from_numpy(x).permute(2, 0, 1)[None].float().to(DEVICE) / 255.0
    f = (torch.from_numpy(ft[min(k, len(ft) - 1)])[None].to(DEVICE)
         if ft is not None else None)
    p, t = model.predict(x, f)       # p(va chạm), TTC giây (inf = không nguy cơ)
    p_hat.append(p)
    ttc_hat.append(t)
    k += 1
cap.release()
model.reset()

p_hat = np.array(p_hat)
ttc_hat = np.array(ttc_hat)
gt = gt[:len(imgs)]

# collision_frame chỉ tồn tại ở schema combined_ego -> đọc bằng .get, không phải
# thuộc tính, nếu không Series sẽ ném AttributeError trên bộ hackathon_ttc
cf = row.get("collision_frame", None)
cf = None if cf is None or (isinstance(cf, float) and np.isnan(cf)) else int(float(cf))

first_warn = next((i for i, v in enumerate(p_hat) if v >= 0.5), None)
first_true = next((i for i, v in enumerate(gt) if 0 <= v <= ck["pos_ttc"]), None)
danger = (gt >= 0) & (gt <= ck["pos_ttc"])

print(f"\n{row.trip_id} · {row.get('dataset', '?')} · {len(imgs)} frame · "
      f"{row.get('width', '?')}x{row.get('height', '?')}")
print(f"nhãn: {int((gt >= 0).sum())}/{len(gt)} frame có TTC hữu hạn · "
      f"{int(danger.sum())} frame nguy hiểm (TTC ≤ {ck['pos_ttc']}s)"
      + (f" · va chạm @ frame {cf}" if cf is not None else ""))
print(f"cảnh báo đầu tiên: model @ frame {first_warn} · nhãn @ frame {first_true}"
      + (f" (sớm/muộn {(first_warn - first_true) / 10:+.1f}s)"
         if first_warn is not None and first_true is not None else ""))
crit = (gt >= 0) & (gt < 3.0)
if crit.any():
    err = np.abs(np.clip(ttc_hat[crit], 0, 10.0) - gt[crit])
    print(f"MAE trên frame TTC<3s: {err.mean():.2f}s ({int(crit.sum())} frame)")

# --- vẽ: nhãn bên trái, dự đoán bên phải, thanh xác suất phía dưới ---------
GREY, OK, WARNC, DANGER = "#9a9a94", "#1baf7a", "#eda100", "#e34948"

def style(v, neg_text="—"):
    if v is None or v < 0:
        return GREY, neg_text
    if not np.isfinite(v):
        return OK, "∞"
    if v == 0:
        return DANGER, "0.0s"
    return (DANGER if v < 0.5 else WARNC if v < 1.5 else OK), f"{v:.1f}s"

W = 640
small = [cv2.resize(im, (W, int(im.shape[0] * W / im.shape[1]))) for im in imgs]
H = small[0].shape[0]

fig, ax = plt.subplots(figsize=(9, 9 * H / W))
ax.axis("off")
fig.subplots_adjust(0, 0, 1, 1)
canvas = ax.imshow(small[0])
box = dict(boxstyle="round,pad=0.4", fc="#111111", ec="none", alpha=0.78)

t_gt = ax.text(0.015, 0.975, "", transform=ax.transAxes, va="top", ha="left",
               fontsize=13, fontweight="bold", bbox=box)
t_pr = ax.text(0.985, 0.975, "", transform=ax.transAxes, va="top", ha="right",
               fontsize=13, fontweight="bold", bbox=box)
t_ft = ax.text(0.5, 0.025, "", transform=ax.transAxes, va="bottom", ha="center",
               fontsize=9, color="white", bbox=dict(box, alpha=0.6))
bar_bg = plt.Rectangle((0.30, 0.90), 0.40, 0.022, transform=ax.transAxes,
                       fc="#3a3a3a", ec="none", zorder=3)
bar_fg = plt.Rectangle((0.30, 0.90), 0.0, 0.022, transform=ax.transAxes,
                       fc=DANGER, ec="none", zorder=4)
ax.add_patch(bar_bg)
ax.add_patch(bar_fg)

def draw(i):
    canvas.set_data(small[i])
    cg, sg = style(float(gt[i]), NEG_TEXT)
    cp, sp = style(float(ttc_hat[i]), "∞")
    t_gt.set_text(f"NHÃN   TTC {sg}")
    t_gt.set_color(cg)
    t_pr.set_text(f"DỰ ĐOÁN  TTC {sp}   p {p_hat[i]:.2f}")
    t_pr.set_color(cp)
    bar_fg.set_width(0.40 * float(p_hat[i]))
    bar_fg.set_color(DANGER if p_hat[i] >= 0.5 else GREY)
    t_ft.set_text(f"{row.trip_id} · frame {i + 1}/{len(small)} · t = {i / 10:.1f}s")
    return canvas, t_gt, t_pr, t_ft, bar_fg

anim = animation.FuncAnimation(fig, draw, frames=len(small), interval=100)
plt.close(fig)
HTML(anim.to_jshtml(fps=10))

In [ ]:
# 9) Chạy trên VIDEO CHẤM ĐIỂM (Hackathon_Dataset_Redacted) — không có nhãn TTC
#
#    10 trip chấm điểm đã bị xoá ground truth (TTC, driver state, risk, vị trí 3D),
#    nên ở đây chỉ hiện DỰ ĐOÁN. Thứ còn lại để đối chiếu là `events_log`: mốc
#    thời gian và loại sự kiện — dùng nó xem model có phản ứng đúng lúc không.
#
#    HAI CHỖ PHẢI KHỚP VỚI LÚC TRAIN:
#      * Tần số: model học trên dữ liệu 10 Hz, bộ này là 20 Hz. TSM lấy kênh từ
#        frame trước và TCN có receptive field 31 frame — cả hai đo bằng ĐƠN VỊ
#        FRAME. Đưa thẳng 20 fps vào thì chuyển động mỗi bước giảm nửa và cửa sổ
#        ngữ cảnh co từ 3.1s còn 1.55s. Vì vậy STRIDE = 2.
#      * Tiền xử lý: resize thẳng 224×224 (méo tỉ lệ y hệt lúc train), RGB, /255,
#        KHÔNG chuẩn hoá mean/std ImageNet.
#
#    Suy luận chạy từ frame 0 để state ấm đúng cách, nhưng chỉ GIỮ ẢNH trong cửa
#    sổ quanh event — 900 frame ảnh màu là ~620 MB RAM, và trình phát cũng không
#    kham nổi.
import gzip, json

TRIP       = "T09d"      # T01d … T10d
EVENT_IDX  = 0           # xem quanh event thứ mấy trong events_log
PRE, POST  = 4.0, 8.0    # giây trước/sau event để hiển thị
STRIDE     = 2           # 20 fps -> 10 fps, khớp lúc train
SAVE_MP4   = True        # ngoài phát trong notebook, ghi luôn ra <repo>/outputs/

HACK = next((p / "datasets" / "Hackathon_Dataset_Redacted" for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "datasets" / "Hackathon_Dataset_Redacted").is_dir()), None)
assert HACK, "không thấy thư mục Hackathon_Dataset_Redacted"
trip_dir = HACK / TRIP

with gzip.open(trip_dir / f"{TRIP}.json.gz", "rt", encoding="utf-8") as fh:
    meta = json.load(fh)
fps_src = meta["metadata"]["fps"]
events = [(e["t"], e["type"]) for e in meta["events_log"]]
ego_speed = {f["frame_id"]: f["ego"]["speed_kmh"] for f in meta["frames"]}
print(f"{TRIP} · {meta['metadata']['map']} · {meta['metadata']['duration_sec']}s "
      f"@{fps_src}fps · limit {meta['metadata']['speed_limit_kmh']} km/h")
print("events:", ", ".join(f"{n}@{t:.0f}s" for t, n in events))

t_ev, ev_name = events[EVENT_IDX]
lo, hi = t_ev - PRE, t_ev + POST

files = sorted((trip_dir / "kitti" / "image_2").glob("*.jpg"))[::STRIDE]
# Feature vô hướng dựng từ CHÍNH JSON của bộ chấm — cùng một hàm với lúc
# train (scripts/infer_hackathon_videos.trip_scalars làm y hệt).
_v = [meta["frames"][i]["ego"]["speed_kmh"] / 3.6 for i in
      [int(f.stem) for f in files]]
_a = [0.0] + [(_v[i] - _v[i - 1]) * 10.0 for i in range(1, len(_v))]
_j = [0.0] + [(_a[i] - _a[i - 1]) * 10.0 for i in range(1, len(_a))]
_w = meta["metadata"].get("weather") or {}
ft = (build_scalars(
        [v * 3.6 for v in _v], _a, _j,
        [meta["frames"][int(f.stem)]["ego"].get("lateral_accel", 0.0) for f in files],
        [len(meta["frames"][int(f.stem)].get("targets") or []) for f in files],
        speed_limit_kmh=meta["metadata"].get("speed_limit_kmh", ""),
        weather={"w_cloud": _w.get("cloudiness", 0.0),
                 "w_rain": _w.get("precipitation", 0.0),
                 "w_wet": max(_w.get("wetness", 0.0),
                              _w.get("precipitation_deposits", 0.0)),
                 "w_fog": _w.get("fog_density", 0.0),
                 "w_sun_alt": _w.get("sun_altitude_angle", 75.0)})
      if model.n_scalar else None)
model.reset()
ts, p_hat, ttc_hat, keep_imgs, keep_idx = [], [], [], [], []
for j, fp in enumerate(files):
    im = cv2.imread(str(fp))
    if im is None:
        continue
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, IMG_SIZE)
    x = torch.from_numpy(x).permute(2, 0, 1)[None].float().to(DEVICE) / 255.0
    f = (torch.from_numpy(ft[min(j, len(ft) - 1)])[None].to(DEVICE)
         if ft is not None else None)
    p, t = model.predict(x, f)
    t_now = int(fp.stem) / fps_src
    ts.append(t_now)
    p_hat.append(p)
    ttc_hat.append(t)
    if lo <= t_now <= hi:                      # chỉ giữ ảnh trong cửa sổ hiển thị
        keep_imgs.append(rgb)
        keep_idx.append(j)
model.reset()

ts, p_hat, ttc_hat = np.array(ts), np.array(p_hat), np.array(ttc_hat)
print(f"\n{len(files)} frame suy luận ({fps_src // STRIDE} fps) · "
      f"p>=0.5 ở {float((p_hat >= 0.5).mean()):.1%} thời lượng")
for t_e, n_e in events:
    w = p_hat[(ts >= t_e - 2) & (ts <= t_e + 10)]
    print(f"  {n_e:24s} @{t_e:5.1f}s → p tối đa trong [-2s,+10s] = "
          f"{(w.max() if len(w) else float('nan')):.2f}")

# --- vẽ: chỉ có dự đoán, kèm nhãn event đang diễn ra -----------------------
GREY, OK, WARNC, DANGER = "#9a9a94", "#1baf7a", "#eda100", "#e34948"

def style(v):
    if not np.isfinite(v):
        return OK, "∞"
    return (DANGER if v < 0.5 else WARNC if v < 1.5 else OK), f"{v:.1f}s"

W = 640
small = [cv2.resize(im, (W, int(im.shape[0] * W / im.shape[1]))) for im in keep_imgs]
H = small[0].shape[0]

fig, ax = plt.subplots(figsize=(9, 9 * H / W))
ax.axis("off")
fig.subplots_adjust(0, 0, 1, 1)
canvas = ax.imshow(small[0])
box = dict(boxstyle="round,pad=0.4", fc="#111111", ec="none", alpha=0.78)
t_pr = ax.text(0.015, 0.975, "", transform=ax.transAxes, va="top", ha="left",
               fontsize=13, fontweight="bold", bbox=box)
t_ev_txt = ax.text(0.985, 0.975, "", transform=ax.transAxes, va="top", ha="right",
                   fontsize=11, fontweight="bold", color="white", bbox=box)
t_ft = ax.text(0.5, 0.025, "", transform=ax.transAxes, va="bottom", ha="center",
               fontsize=9, color="white", bbox=dict(box, alpha=0.6))
bar_bg = plt.Rectangle((0.30, 0.90), 0.40, 0.022, transform=ax.transAxes,
                       fc="#3a3a3a", ec="none", zorder=3)
bar_fg = plt.Rectangle((0.30, 0.90), 0.0, 0.022, transform=ax.transAxes,
                       fc=DANGER, ec="none", zorder=4)
ax.add_patch(bar_bg)
ax.add_patch(bar_fg)

def draw(i):
    j = keep_idx[i]
    canvas.set_data(small[i])
    cp, sp = style(float(ttc_hat[j]))
    t_pr.set_text(f"DỰ ĐOÁN  TTC {sp}   p {p_hat[j]:.2f}")
    t_pr.set_color(cp)
    dt = ts[j] - t_ev
    t_ev_txt.set_text(f"{ev_name}  {dt:+.1f}s" if dt >= 0 else f"{ev_name} sắp tới {dt:+.1f}s")
    bar_fg.set_width(0.40 * float(p_hat[j]))
    bar_fg.set_color(DANGER if p_hat[j] >= 0.5 else GREY)
    spd = ego_speed.get(int(files[j].stem), float("nan"))
    t_ft.set_text(f"{TRIP} · frame {files[j].stem} · t = {ts[j]:.1f}s · {spd:.0f} km/h")
    return canvas, t_pr, t_ev_txt, t_ft, bar_fg


# --- lưu ra file mp4 -------------------------------------------------------
#    KHÔNG dùng anim.save(): nó cần ffmpeg trên PATH, mà máy này chỉ có writer
#    'pillow' và 'html' (kiểm bằng animation.writers.list()) — pillow chỉ ra gif,
#    nặng gấp nhiều lần. Cách dưới render từng frame của figure rồi đẩy qua
#    cv2.VideoWriter, xài ffmpeg đã bundled sẵn trong opencv: không thêm phụ
#    thuộc, và ra thẳng thứ dán được vào slide.
def save_mp4(fig, draw_fn, n_frames, path, fps=10):
    """Render figure -> mp4 bằng cv2. Dùng lại được cho cell 8 (video có nhãn)."""
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    w, h = w - w % 2, h - h % 2        # codec đòi cạnh chẵn; lẻ thì file hỏng im lặng
    path.parent.mkdir(parents=True, exist_ok=True)
    vw = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (w, h))
    assert vw.isOpened(), f"không mở được VideoWriter cho {path}"
    try:
        for i in range(n_frames):
            draw_fn(i)
            fig.canvas.draw()
            vw.write(cv2.cvtColor(np.asarray(fig.canvas.buffer_rgba())[:h, :w, :3],
                                  cv2.COLOR_RGB2BGR))
    finally:
        vw.release()                   # thiếu release() -> file thiếu moov atom, không phát được
    return path


if SAVE_MP4:
    out_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "datasets").is_dir()), Path.cwd())
    dst = save_mp4(fig, draw, len(small),
                   out_root / "outputs" / f"{TRIP}_ev{EVENT_IDX}_{ev_name}.mp4")
    print(f"\nđã lưu {dst} ({dst.stat().st_size / 1e6:.1f} MB)")

anim = animation.FuncAnimation(fig, draw, frames=len(small), interval=100)
plt.close(fig)
HTML(anim.to_jshtml(fps=10))
